# 实验 23 — dbt incremental 策略深入

**目标**：实验 6 入门了 incremental，这次把 4 种常见策略都过一遍，知道什么时候选哪个。

## 四种 incremental_strategy

| 策略 | 适用 | 行为 |
|---|---|---|
| `append` | 不变事实流（埋点日志） | 直接 insert 新行，不去重 |
| `merge` | 有自然主键、上游可能修正 | upsert（按 `unique_key`），最通用 |
| `delete+insert` | 类似 merge，但事务里先 delete 主键再 insert | DuckDB / Postgres 兼容性更好 |
| `insert_overwrite` | 大表 + 分区列 | 按 partition 删除后写入，效率高（BigQuery / Spark 常用） |

注意：可用策略由 adapter 决定。dbt-duckdb 支持 `append` / `merge` / `delete+insert`。

In [ ]:
# 建一个 incremental fact 模型做实验
from pathlib import Path
import subprocess, os, duckdb

model = Path('../dbt_project/models/marts/fct_daily_rates_inc.sql')
model.write_text("""{{ config(
    materialized='incremental',
    unique_key=['rate_date','currency'],
    incremental_strategy='merge'
) }}

select rate_date, currency, rate
from {{ ref('stg_ecb_rates') }}
{% if is_incremental() %}
where rate_date > (select coalesce(max(rate_date), '1900-01-01') from {{ this }})
{% endif %}
""")

def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

print('=== 首次 build（全量）===')
print(dbt('run','--select','fct_daily_rates_inc')[-800:])

In [ ]:
con = duckdb.connect('../data/sandbox.duckdb')
print('rows:', con.execute('select count(*) from main.fct_daily_rates_inc').fetchone()[0])
print('max date:', con.execute('select max(rate_date) from main.fct_daily_rates_inc').fetchone()[0])

In [ ]:
# === merge 策略：先看编译后 SQL ===
print(dbt('compile','--select','fct_daily_rates_inc')[-200:])
p = Path('../dbt_project/target/run/dlt_dbt_sandbox/models/marts/fct_daily_rates_inc.sql')
# 增量再跑一次，看 run/ 下的 SQL（是真正发给 DB 的 merge into ...）
print(dbt('run','--select','fct_daily_rates_inc')[-300:])
print()
if p.exists():
    print('=== 实际执行的 SQL（截断）===')
    print(p.read_text()[:1500])

In [ ]:
# === 切到 delete+insert 策略 ===
model.write_text(model.read_text().replace("incremental_strategy='merge'",
                                           "incremental_strategy='delete+insert'"))
print(dbt('run','--select','fct_daily_rates_inc')[-300:])
print()
print(Path('../dbt_project/target/run/dlt_dbt_sandbox/models/marts/fct_daily_rates_inc.sql').read_text()[:1500])

In [ ]:
# === append 策略 + on_schema_change ===
# append 不去重——如果同一个 rate_date 跑两次，会有重复行（除非你的 where 过滤干净）
model.write_text("""{{ config(
    materialized='incremental',
    incremental_strategy='append',
    on_schema_change='append_new_columns'
) }}
select rate_date, currency, rate, current_timestamp as ingested_at
from {{ ref('stg_ecb_rates') }}
{% if is_incremental() %}
where rate_date > (select max(rate_date) from {{ this }})
{% endif %}
""")
print(dbt('run','--select','fct_daily_rates_inc','--full-refresh')[-300:])
# 再跑一次模拟“同一天再来一批”
print(dbt('run','--select','fct_daily_rates_inc')[-300:])
print('rows now:', con.execute('select count(*) from main.fct_daily_rates_inc').fetchone()[0])

In [ ]:
# 清理
import os
os.remove('../dbt_project/models/marts/fct_daily_rates_inc.sql')
con.execute('drop table if exists main.fct_daily_rates_inc')

## 生产环境踩坑提示

- **`merge` 默认要 `unique_key`**。可以是单列也可以是 `[col1, col2]`。漏配会让 dbt fall back 到 `append` —— 沉默地翻车。
- **`on_schema_change`**：
  - `ignore`（默认，1.6 之后是 `append_new_columns`）：列对不上忽略
  - `sync_all_columns`：drop 旧列、add 新列
  - `append_new_columns`：只加不删（最常见的生产选择）
  - `fail`：直接挂掉强制人审
- **incremental 的 `where`** 是性能关键。`where rate_date > (select max(rate_date) from {{ this }})` 会触发对结果表的扫描——大表用 `dbt_utils.dateadd` + 写死 lookback 更稳。
- **lookback window**：上游有延迟（晚到 2 天）时，单纯 `> max(rate_date)` 会漏数据。改成 `>= dateadd(day, -3, current_date)` + `merge` 处理重复。
- **`--full-refresh` 在生产**：定期（每月）跑一次防止漂移；CI 上别默认开（成本爆炸）。
- **`insert_overwrite` + `partition_by`** 是大数据量首选（BigQuery / Spark / Snowflake clustering）。DuckDB 这种 OLAP 单机引擎用不到。

## 思考题

- `merge` vs `delete+insert` 在事务语义上有什么区别？哪个更安全？
- 把 `unique_key` 改成单列 `rate_date`，merge 怎么处理 `(2024-01-01, USD)` 和 `(2024-01-01, GBP)`？（提示：会撞，只剩一行）
- 真实的 “late-arriving fact” 场景：上游 2024-01-01 的修正 3 天后才到，你要怎么改 incremental where 子句？